In [ ]:
#| default_exp menus

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *

A host describes its menu bar as data: a list of menus, each with a title and a list of rows. This module defines the row types, and the small part of a keymap a menu needs to read. They are plain dataclasses that do not import pywebview or AppKit, so a host can build and test its menu table on any machine. `kavacha.window` turns the table into a native menu bar on macOS.

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass

### The rows

`MenuItem` is a row that runs one of the host's actions, named by `action`. When the user chooses it, the window calls the host's `on_action` with that name. A `MenuItem` with no action is a separator line, which is why `action` has a default. `kind` is either empty or `warn`. `warn` marks a row the host may want to draw as destructive, such as one that discards changes; kavacha passes it through without acting on it, and refuses any other value.

In [ ]:
#| export
@dataclass(frozen=True)
class MenuItem:
    "One action in a menu. A blank action is a separator."
    action: str = ''
    kind: str = ''
    def __post_init__(self):
        if self.kind not in ('', 'warn'):
            raise ValueError(f'{self.action or "separator"}: bad menu kind {self.kind!r}')

In [ ]:
MenuItem('open_folder'), MenuItem()

In [ ]:
#| hide
test_eq(MenuItem().action, '')                     
test_eq(MenuItem('git_discard', 'warn').kind, 'warn')
test_fail(lambda: MenuItem('x', 'loud'), contains='bad menu kind')

`Std` is a row that AppKit already knows how to handle, such as Cut, Copy, Paste and Select All, so the host does not handle it at all. The row names an AppKit selector, such as `cut:`. The menu sends that selector along AppKit's responder chain, the chain of objects that starts at whatever has keyboard focus, and the first one that understands it acts. `key` and `mods` give the row's shortcut directly, because a row like this has no entry in the host's keymap.

`Js` is a row that runs a JavaScript expression in the focused window, for small jobs such as reloading the page that need no action of their own.

In [ ]:
#| export
@dataclass(frozen=True)
class Std:
    "A row AppKit handles itself, by sending `selector` to whatever has focus."
    title: str
    selector: str
    key: str = ''
    mods: str = 'mod'

In [ ]:
#| export
@dataclass(frozen=True)
class Js:
    "A row that runs the JavaScript expression `expr` in the focused window."
    title: str
    expr: str

In [ ]:
Std('Cut', 'cut:', 'x'), Js('Reload', 'location.reload()')

### Key bindings

A menu shows each action's label and shortcut, and the host keeps both in its keymap. kavacha does not read the keymap itself. The host supplies a lookup function that takes an action name and returns a `Binding`, or `None` for an action it does not know. A `Binding` holds the action's chords, its label, and its scope, meaning where the binding applies. A `global` binding becomes a menu shortcut. A binding with another scope becomes one only when `shared` is false, which the host sets when no other scope binds the same chord. A dict's `get` method is the simplest lookup, as in the example.

In [ ]:
#| export
@dataclass(frozen=True)
class Binding:
    "What a menu row reads from the host's keymap: the chords, the label, where it applies, and whether another scope binds its chord."
    keys: tuple = ()
    label: str = ''
    scope: str = 'global'
    shared: bool = False
    @property
    def bound(self): return bool(self.keys)

In [ ]:
keys = {'save': Binding(('mod+s',), 'Save'), 'find': Binding(('mod+f',), scope='editor')}
keys.get('save'), keys.get('nothing')

`bound` is true when the binding has at least one chord. It is worked out from `keys` instead of being stored, so an unbound action is just `Binding()` and the two can never disagree.

In [ ]:
#| hide
test_eq(Binding().bound, False)
test_eq(Binding(('mod+s',)).bound, True)
test_eq(Binding(('mod+s',)).scope, 'global')
test_eq(Binding(('mod+s',)).shared, False)
test_eq(keys.get('save').keys[0], 'mod+s')
test_is(keys.get('nothing'), None)

### Chords a menu must not take

A menu shortcut wins over the web page, so a chord the system or a window manager already uses must never become one. `RESERVED` lists them in the keymap grammar: Hyper (`hyper`, Command, Control, Option and Shift together) with the keys AeroSpace takes, AeroSpace's own Control and Option chords, and the macOS system chords. `kavacha.window.menu_chord` refuses every one of them, and a host can lint its default keymap against the same set.

In [ ]:
#| export
_AERO_HYPER = [*'123456789', 'tab', *'sqawhjklrftcd', 'space', *'`[]=-0', 'up', 'down', 'left', 'right']
RESERVED = frozenset([*(f'hyper+{k}' for k in _AERO_HYPER),
    *(f'ctrl+alt+shift+{k}' for k in [*'123456789', 'left', 'right']),
    *(f'ctrl+alt+{k}' for k in [*'123456789', *'hjkl', 'up', 'down', 'left', 'right']),
    'alt+shift+left', 'alt+shift+right', 'alt+shift+;', 'mod+ctrl+tab', 'mod+ctrl+`',
    'ctrl+space', 'ctrl+up', 'ctrl+down', 'ctrl+left', 'ctrl+right', 'mod+space', 'mod+tab', 'mod+`',
    'mod+q', 'mod+h', 'mod+alt+h', 'mod+m', 'mod+alt+d', 'mod+alt+escape', 'mod+shift+3', 'mod+shift+4',
    'mod+shift+5', 'mod+ctrl+q', 'mod+ctrl+space', 'mod+ctrl+f'])

In [ ]:
#| hide
test_eq('hyper+s' in RESERVED, True)
test_eq('hyper+n' in RESERVED, False)
test_eq({'mod+q', 'ctrl+alt+h', 'alt+shift+;'} <= RESERVED, True)

The launcher's own rows are `Std` rows every Mac app has. `LAUNCHER_ROWS` lists them with their chords, so a host can check that no chord on its own menu bar lands on one of them. Close Window takes ⇧⌘W, which leaves ⌘W for the host's Close Tab.

In [ ]:
#| export
LAUNCHER_ROWS = (Std('Quit', 'terminate:', 'q'), Std('Hide', 'hide:', 'h'),
                 Std('Hide Others', 'hideOtherApplications:', 'h', 'mod+alt'),
                 Std('Close Window', 'performClose:', 'w', 'mod+shift'), Std('Undo', 'undo:', 'z'),
                 Std('Redo', 'redo:', 'z', 'mod+shift'), Std('Minimize', 'performMiniaturize:', 'm'),
                 Std('Enter Full Screen', 'toggleFullScreen:', 'f', 'mod+ctrl'))

In [ ]:
#| hide
test_eq([r.title for r in LAUNCHER_ROWS if r.selector == 'performClose:'], ['Close Window'])
test_eq(len({(r.key, r.mods) for r in LAUNCHER_ROWS}), len(LAUNCHER_ROWS))

### How a chord is written for a person

`chord_glyph` writes one chord the way a Mac menu does: the modifier glyphs in the order ⌃⌥⇧⌘, then the key as a capital letter or a symbol such as ↩ or ⌫. With `mac=False` it writes `Ctrl+Shift+P`. `mod` is Command on a Mac and Control elsewhere, `hyper` is all four modifiers, and a capital letter means Shift. A sequence of chords separated by spaces keeps its spaces. Every place a host shows a chord, such as its menus, palette, help and toasts, can use this one function, so they all agree.

In [ ]:
#| export
_GLYPH = {'enter': '↩', 'return': '↩', 'backspace': '⌫', 'delete': '⌦', 'escape': '⎋', 'esc': '⎋', 'tab': '⇥',
          'space': 'Space', 'up': '↑', 'down': '↓', 'left': '←', 'right': '→', 'pageup': '⇞', 'pagedown': '⇟',
          'home': '↖', 'end': '↘'}
_PC = {'enter': 'Enter', 'return': 'Enter', 'backspace': 'Backspace', 'delete': 'Delete', 'escape': 'Esc', 'esc': 'Esc',
       'tab': 'Tab', 'space': 'Space', 'up': 'Up', 'down': 'Down', 'left': 'Left', 'right': 'Right',
       'pageup': 'PageUp', 'pagedown': 'PageDown', 'home': 'Home', 'end': 'End'}
_ALIAS = {'cmd': 'mod', 'opt': 'alt'}

def _glyph(chord, mac):
    *mods, base = chord.split('+') if not chord.endswith('++') else [*chord[:-2].split('+'), '+']
    ms = {_ALIAS.get(m.lower(), m.lower()) for m in mods if m}
    if 'hyper' in ms: ms = (ms - {'hyper'}) | {'mod', 'ctrl', 'alt', 'shift'}
    if len(base) == 1 and base.isupper(): ms.add('shift')
    b = base.lower()
    if mac: return ''.join(g for m, g in (('ctrl', '⌃'), ('alt', '⌥'), ('shift', '⇧'), ('mod', '⌘')) if m in ms) + _GLYPH.get(b, b.upper())
    names = [n for m, n in (('mod', 'Ctrl'), ('ctrl', 'Ctrl'), ('alt', 'Alt'), ('shift', 'Shift')) if m in ms]
    return '+'.join([*dict.fromkeys(names), _PC.get(b, b.upper())])

def chord_glyph(chord, mac=True):
    "How `chord` is written for a person: ⌃⌥⇧⌘ and a capital on a Mac, `Ctrl+Shift+P` elsewhere."
    return ' '.join(_glyph(c, mac) for c in str(chord or '').split())

In [ ]:
#| exec_doc
chord_glyph('mod+shift+p'), chord_glyph('alt+shift+b'), chord_glyph('mod+enter'), chord_glyph('mod+shift+p', mac=False)

In [ ]:
#| hide
test_eq(chord_glyph('mod+shift+p'), '⇧⌘P')
test_eq(chord_glyph('alt+shift+b'), '⌥⇧B')
test_eq(chord_glyph('ctrl+alt+shift+mod+x'), chord_glyph('hyper+x'))
test_eq(chord_glyph('hyper+x'), '⌃⌥⇧⌘X')
test_eq(chord_glyph('mod+P'), '⇧⌘P')
test_eq(chord_glyph('mod+alt+backspace'), '⌥⌘⌫')
test_eq(chord_glyph('ctrl+`'), '⌃`')
test_eq(chord_glyph('mod+shift+/'), '⇧⌘/')
test_eq(chord_glyph('d d'), 'D D')
test_eq(chord_glyph('mod+enter', mac=False), 'Ctrl+Enter')
test_eq(chord_glyph('mod+ctrl+f', mac=False), 'Ctrl+F')
test_eq(chord_glyph(''), '')

`kavacha.window.menu_bar` takes the menu table and the lookup function together. kavacha ships no default table for either, because what belongs on an app's File menu is up to the app.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()